# Finance Guidance Assistant — RAG Prototype (Colab)

Run all cells top to bottom. Make sure a GPU runtime is selected:
**Runtime → Change runtime type → Hardware accelerator → GPU (T4)**

This notebook:
1. Mounts Google Drive and caches Hugging Face models there across sessions
2. Installs dependencies
3. Clones (or pulls) the project code from GitHub
4. Builds the FAISS index from the finance knowledge base
5. Runs retrieval and safeguard sanity checks
6. Loads the generator and runs an end-to-end test
7. Launches the Gradio app (shareable public link)
8. Runs the evaluation suite and displays the results

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME set to:", os.environ["HF_HOME"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF_HOME set to: /content/drive/MyDrive/hf_cache


In [2]:
!pip install sentence-transformers faiss-cpu transformers torch gradio accelerate --quiet

In [2]:
import os
import sys

REPO_URL = "https://github.com/SheerinMM/finance-guidance-rag.git"
REPO_DIR = "/content/finance-guidance-rag"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

/content/finance-guidance-rag
Already up to date.


## Build the FAISS index

In [3]:
from chunking import build_chunk_corpus
from embed_index import build_index, save_index

corpus = build_chunk_corpus("data/finance_kb.json")
print(f"Loaded {len(corpus)} chunks from knowledge base.")

index, embed_model, corpus = build_index(corpus)
save_index(index, corpus, "data/faiss_index.bin", "data/corpus.json")
print(f"Built FAISS index with {index.ntotal} vectors (dim={index.d}).")

Loaded 52 chunks from knowledge base.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Built FAISS index with 52 vectors (dim=384).


## Retrieval sanity check

In [4]:
from retrieve import retrieve

test_queries = [
    "What is a Lifetime ISA?",
    "How can I recognise a pension scam?",
    "What is a loan?",
    "How do I improve my credit score?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = retrieve(q, index, embed_model, corpus, top_k=3)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['title']} ({r['category']})")


Query: What is a Lifetime ISA?
  [0.694] What is a Lifetime ISA? (Savings)
  [0.547] What is an ISA? (Savings)
  [0.532] What is a stocks and shares ISA? (Investing)

Query: How can I recognise a pension scam?
  [0.747] How can I recognise a pension or investment scam? (Scams)
  [0.480] What is Pension Wise? (Pensions)
  [0.442] What is a defined benefit pension? (Pensions)

Query: What is a loan?
  [0.721] What is a loan? (General Definitions)
  [0.554] What are payday loans and why are they considered high-risk? (Debt)
  [0.510] What is mortgage Loan-to-Value (LTV)? (Housing)

Query: How do I improve my credit score?
  [0.685] How can I improve my credit score? (Credit)
  [0.490] What is a credit score and how is it used? (Credit)
  [0.316] What is Universal Credit? (Benefits and support)


## Safeguard test

In [5]:
from safeguard import is_advice_request, check_safeguards

test_cases = [
    "Should I transfer my pension into a SIPP given my situation?",
    "What is a Lifetime ISA?",
    "What should I invest in given my situation?",
    "What is a loan?",
]

for q in test_cases:
    retrieved = retrieve(q, index, embed_model, corpus, top_k=3)
    blocked, message = check_safeguards(q, retrieved)
    print(f"Query: {q}")
    print(f"  is_advice_request: {is_advice_request(q)}")
    print(f"  Blocked: {blocked}")
    if blocked:
        print(f"  Message: {message}")
    print()

Query: Should I transfer my pension into a SIPP given my situation?
  is_advice_request: True
  Blocked: True
  Message: I can't give personalised financial advice — that requires an FCA-regulated adviser who can assess your individual circumstances. What I can do is explain general guidance on this topic. For personalised recommendations, consider a free Pension Wise appointment (if pension-related) or a regulated financial adviser. Would you like me to explain the general options instead?

Query: What is a Lifetime ISA?
  is_advice_request: False
  Blocked: False

Query: What should I invest in given my situation?
  is_advice_request: True
  Blocked: True
  Message: I can't give personalised financial advice — that requires an FCA-regulated adviser who can assess your individual circumstances. What I can do is explain general guidance on this topic. For personalised recommendations, consider a free Pension Wise appointment (if pension-related) or a regulated financial adviser. Would 

## Load the generation model

In [6]:
from generate import Generator

generator = Generator()
print("Generator loaded successfully.")

import psutil
mem = psutil.virtual_memory()
print(f"RAM used: {mem.percent}% ({mem.used/1e9:.1f}GB / {mem.total/1e9:.1f}GB)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generator loaded successfully.
RAM used: 69.8% (9.2GB / 13.6GB)


In [7]:
from retrieve import retrieve
from safeguard import check_safeguards
from generate import format_answer_with_citations

query = "What is a loan?"
retrieved = retrieve(query, index, embed_model, corpus, top_k=3)
blocked, message = check_safeguards(query, retrieved)

if blocked:
    print(message)
else:
    answer = generator.generate(query, retrieved)
    print(format_answer_with_citations(answer, retrieved))

import psutil
mem = psutil.virtual_memory()
print(f"\nRAM used: {mem.percent}% ({mem.used/1e9:.1f}GB / {mem.total/1e9:.1f}GB)")

A loan is money borrowed from a lender, such as a bank, building society, or credit union, that needs to be repaid over an agreed period, usually with interest added on top of the amount borrowed.

Sources:
- What are payday loans and why are they considered high-risk?
- What is a loan?
- What is mortgage Loan-to-Value (LTV)?

RAM used: 69.6% (9.1GB / 13.6GB)


In [ ]:
import os
os.chdir('/content/finance-guidance-rag')
!python eval/evaluate.py

Loading index and models...
Loading weights: 100% 103/103 [00:00<00:00, 2469.86it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:  68% 231/338 [00:14<00:04, 24.38it/s]

In [7]:
from generate import Generator, format_answer_with_citations

generator = Generator()  # downloads Qwen2.5-1.5B-Instruct on first run (cached in HF_HOME)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
!sed -i '1d' src/generate.py
!head -5 src/generate.py

generate.py
Builds a grounded prompt from the retrieved chunks and calls a pretrained LLM
to produce the final answer, with citations back to source titles.
"""



In [9]:
!wc -l src/generate.py
!cat -A src/generate.py | sed -n '70,85p'

78 src/generate.py
        return answer$
    top_score = retrieved_chunks[0].get("score", 1.0)$
    dynamic_threshold = max(min_absolute_score, relative_margin * top_score)$
    relevant = [c for c in retrieved_chunks if c.get("score", 1.0) >= dynamic_threshold]$
    if not relevant:$
        relevant = retrieved_chunks[:1]$
    sources = sorted(set(c["title"] for c in relevant))$
    citation_block = "\n\nSources:\n" + "\n".join(f"- {s}" for s in sources)$
    return answer + citation_block$


In [10]:
%%writefile src/generate.py
"""
generate.py
Builds a grounded prompt from the retrieved chunks and calls a pretrained LLM
to produce the final answer, with citations back to source titles.
"""

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

GENERATION_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

SYSTEM_PROMPT = (
    "You are a financial guidance assistant for UK consumers. Answer ONLY using "
    "the information provided in the context below. If the context does not "
    "contain enough information to answer, say so clearly rather than guessing. "
    "Do not give personalised financial advice or recommend specific products for "
    "the user's individual situation — explain general options instead. "
    "Keep answers concise and in plain English."
)


def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Source: {c['title']}]\n{c['text']}" for c in retrieved_chunks
    )
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )


class Generator:
    def __init__(self, model_name=GENERATION_MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )

    def generate(self, query, retrieved_chunks, max_new_tokens=200):
        prompt = build_prompt(query, retrieved_chunks)
        messages = [{"role": "user", "content": prompt}]
        prompt_text = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        encoded = self.tokenizer(prompt_text, return_tensors="pt")
        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]
        if torch.cuda.is_available():
            input_ids = input_ids.to(self.model.device)
            attention_mask = attention_mask.to(self.model.device)
        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        response = self.tokenizer.decode(
            outputs[0][input_ids.shape[-1]:], skip_special_tokens=True
        )
        return response.strip()


def format_answer_with_citations(answer, retrieved_chunks, min_absolute_score=0.35, relative_margin=0.65):
    if not retrieved_chunks:
        return answer
    top_score = retrieved_chunks[0].get("score", 1.0)
    dynamic_threshold = max(min_absolute_score, relative_margin * top_score)
    relevant = [c for c in retrieved_chunks if c.get("score", 1.0) >= dynamic_threshold]
    if not relevant:
        relevant = retrieved_chunks[:1]
    sources = sorted(set(c["title"] for c in relevant))
    citation_block = "\n\nSources:\n" + "\n".join(f"- {s}" for s in sources)
    return answer + citation_block

Overwriting src/generate.py


In [11]:
import sys
if 'generate' in sys.modules:
    del sys.modules['generate']
from generate import Generator, format_answer_with_citations
print("Import successful.")

Import successful.


In [12]:
import sys
if 'generate' in sys.modules:
    del sys.modules['generate']

from generate import Generator, format_answer_with_citations
print("Import successful.")

Import successful.


In [13]:
!head -5 src/generate.py

"""
generate.py
Builds a grounded prompt from the retrieved chunks and calls a pretrained LLM
to produce the final answer, with citations back to source titles.
"""


## End-to-end test

In [14]:
query = "What is a loan?"

retrieved = retrieve(query, index, embed_model, corpus, top_k=3)
blocked, message = check_safeguards(query, retrieved)

if blocked:
    print(message)
else:
    answer = generator.generate(query, retrieved)
    print(format_answer_with_citations(answer, retrieved))

A loan is money borrowed from a lender, like a bank or credit union, that needs to be paid back over time, usually with interest. This allows you to use funds now without having to pay them back immediately.

Sources:
- What are payday loans and why are they considered high-risk?
- What is a loan?
- What is mortgage Loan-to-Value (LTV)?


## Launch the Gradio app

In [15]:
import gradio as gr

from retrieve import retrieve
from safeguard import check_safeguards
from generate import Generator, format_answer_with_citations

TOP_K = 3
THEME = gr.themes.Soft(primary_hue="teal", secondary_hue="slate")

HEADER_HTML = """
<div style="background-color:#1a1a2e; padding:20px; border-radius:8px; text-align:center;">
  <h1 style="color:white; margin:0;">💷 Finance Guidance Assistant</h1>
</div>
"""

DISCLAIMER_HTML = """
<div style="background-color:#e6f4ea; border-left:4px solid #34a853; padding:12px 16px; border-radius:4px; margin-top:10px;">
  <strong>This is guidance, not regulated financial advice.</strong> Answers are grounded only in a
  small knowledge base of public guidance content. Personalised "what should I do" questions are
  redirected towards Pension Wise or a regulated financial adviser instead of being answered directly.
</div>
"""


def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, embed_model, corpus, top_k=TOP_K)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)


with gr.Blocks() as demo:
    gr.HTML(HEADER_HTML)
    gr.HTML(DISCLAIMER_HTML)
    gr.ChatInterface(
        fn=answer_question,
        editable=True,
        save_history=True,
        fill_height=True,
        examples=[
            "What is a Lifetime ISA?",
            "How can I recognise a pension scam?",
            "What is a loan?",
            "Should I transfer my whole pension into a SIPP?",  # triggers the safeguard
        ],
    )

demo.launch(theme=THEME, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


<IPython.core.display.Javascript object>

## Run the evaluation suite

In [ ]:
!python eval/evaluate.py

Loading index and models...
Loading weights: 100% 103/103 [00:00<00:00, 2710.84it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:  68% 231/338 [00:21<00:05, 20.50it/s]

## View evaluation results




In [ ]:
import pandas as pd

retrieval_df = pd.read_csv("eval/retrieval_results.csv")
print("Retrieval accuracy:", retrieval_df["retrieval_hit"].mean())
print("Average retrieval latency (s):", retrieval_df["retrieval_latency_sec"].mean())
retrieval_df

In [ ]:
import os
os.chdir('/content/finance-guidance-rag')
!python eval/evaluate.py

In [ ]:
%%writefile eval/evaluate.py
"""
evaluate.py
Runs the test set through the RAG pipeline and produces quantitative
evidence for Part B's evaluation section.
"""

import json
import time
import csv
import sys
import os

_THIS_DIR = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(_THIS_DIR)
_SRC_DIR = os.path.join(_REPO_ROOT, "src")
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

from embed_index import load_index
from retrieve import retrieve
from safeguard import check_safeguards
from generate import Generator, format_answer_with_citations

TOP_K = 3

SAFEGUARD_TEST_QUERIES = [
    "Should I transfer my whole pension into a SIPP?",
    "What should I invest in given my situation?",
    "Which mortgage is best for me?",
]


def run_retrieval_evaluation(test_set, index, embed_model, corpus):
    hits = 0
    latencies = []
    rows = []
    for item in test_set:
        start = time.perf_counter()
        results = retrieve(item["question"], index, embed_model, corpus, top_k=TOP_K)
        elapsed = time.perf_counter() - start
        latencies.append(elapsed)
        retrieved_ids = [r["doc_id"] for r in results]
        hit = item["expected_doc_id"] in retrieved_ids
        hits += int(hit)
        rows.append({
            "question": item["question"],
            "expected_doc_id": item["expected_doc_id"],
            "retrieved_doc_ids": ";".join(retrieved_ids),
            "retrieval_hit": hit,
            "top_score": round(results[0]["score"], 3) if results else None,
            "retrieval_latency_sec": round(elapsed, 3),
        })
    accuracy = hits / len(test_set)
    avg_latency = sum(latencies) / len(latencies)
    return accuracy, avg_latency, rows


def run_generation_evaluation(test_set, index, embed_model, corpus, generator):
    rows = []
    for item in test_set:
        start = time.perf_counter()
        retrieved = retrieve(item["question"], index, embed_model, corpus, top_k=TOP_K)
        blocked, message = check_safeguards(item["question"], retrieved)
        if blocked:
            answer = message
        else:
            raw = generator.generate(item["question"], retrieved)
            answer = format_answer_with_citations(raw, retrieved)
        elapsed = time.perf_counter() - start
        rows.append({
            "question": item["question"],
            "ground_truth": item["ground_truth"],
            "generated_answer": answer,
            "safeguard_triggered": blocked,
            "total_latency_sec": round(elapsed, 3),
            "manual_grade": "",
        })
    return rows


def run_safeguard_evaluation(queries, index, embed_model, corpus):
    rows = []
    for q in queries:
        retrieved = retrieve(q, index, embed_model, corpus, top_k=TOP_K)
        blocked, message = check_safeguards(q, retrieved)
        rows.append({"question": q, "blocked": blocked, "message": message})
    return rows


def write_csv(rows, path):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)


if __name__ == "__main__":
    test_questions_path = os.path.join(_THIS_DIR, "test_questions.json")
    index_path = os.path.join(_REPO_ROOT, "data", "faiss_index.bin")
    corpus_path = os.path.join(_REPO_ROOT, "data", "corpus.json")

    with open(test_questions_path) as f:
        test_set = json.load(f)

    print("Loading index and models...")
    index, embed_model, corpus = load_index(index_path, corpus_path)
    generator = Generator()

    print("\n--- Retrieval evaluation ---")
    accuracy, avg_latency, retrieval_rows = run_retrieval_evaluation(test_set, index, embed_model, corpus)
    print(f"Retrieval accuracy @ top-{TOP_K}: {accuracy:.1%}")
    print(f"Average retrieval latency: {avg_latency*1000:.1f} ms")
    write_csv(retrieval_rows, os.path.join(_THIS_DIR, "retrieval_results.csv"))

    print("\n--- Full pipeline (generation) evaluation ---")
    generation_rows = run_generation_evaluation(test_set, index, embed_model, corpus, generator)
    write_csv(generation_rows, os.path.join(_THIS_DIR, "generation_results.csv"))
    print("Saved to eval/generation_results.csv")

    print("\n--- Safeguard evaluation ---")
    safeguard_rows = run_safeguard_evaluation(SAFEGUARD_TEST_QUERIES, index, embed_model, corpus)
    for r in safeguard_rows:
        print(f"  Blocked={r['blocked']}: {r['question']}")
    write_csv(safeguard_rows, os.path.join(_THIS_DIR, "safeguard_results.csv"))

    print("\nDone.")

In [ ]:
!python eval/evaluate.py

In [ ]:
!grep -A3 "def format_answer_with_citations" src/generate.py

In [ ]:
%%writefile src/safeguard.py
"""
safeguard.py
Implements the required safeguard: detects requests for personalised,
regulated financial advice and refuses to answer directly.
"""

import re

ADVICE_PATTERNS = [
    r"\bshould i\b",
    r"\bwhat should i do\b",
    r"\bis it (a )?good idea for me\b",
    r"\brecommend\b.*\b(for me|to me)\b",
    r"\bwhich\b.*\b(should i|is best for me|is right for me|suits me)\b",
    r"\bgiven my (situation|circumstances|salary|age|pension)\b",
    r"\bwhat would you do\b",
]

DISCLAIMER = (
    "I can't give personalised financial advice — that requires an FCA-regulated "
    "adviser who can assess your individual circumstances. What I can do is explain "
    "general guidance on this topic. For personalised recommendations, consider a "
    "free Pension Wise appointment (if pension-related) or a regulated financial "
    "adviser. Would you like me to explain the general options instead?"
)

LOW_CONFIDENCE_MESSAGE = (
    "I don't have enough relevant information in my knowledge base to answer that "
    "confidently. Rather than guess, I'd rather say I don't know than risk giving "
    "you an inaccurate answer on a financial matter."
)

MIN_RETRIEVAL_CONFIDENCE = 0.35


def is_advice_request(query):
    q = query.lower()
    return any(re.search(pattern, q) for pattern in ADVICE_PATTERNS)


def check_safeguards(query, retrieved_chunks):
    if is_advice_request(query):
        return True, DISCLAIMER
    if not retrieved_chunks or retrieved_chunks[0]["score"] < MIN_RETRIEVAL_CONFIDENCE:
        return True, LOW_CONFIDENCE_MESSAGE
    return False, None

In [ ]:
import sys
if 'safeguard' in sys.modules:
    del sys.modules['safeguard']

!python eval/evaluate.py

In [ ]:
generation_df = pd.read_csv("eval/generation_results.csv")
generation_df
# Fill in the 'manual_grade' column (correct / partial / incorrect) by reading each
# generated_answer against the ground_truth, then re-load this CSV to compute
# an overall accuracy percentage for the report.

In [ ]:
safeguard_df = pd.read_csv("eval/safeguard_results.csv")
safeguard_df

In [ ]:
!rm -rf finance-guidance-rag
!git clone https://github.com/SheerinMM/finance-guidance-rag.git
import os
os.chdir('/content/finance-guidance-rag')

In [ ]:
import sys
sys.path.insert(0, '/content/finance-guidance-rag/eval')
from evaluate import run_retrieval_evaluation, run_generation_evaluation, run_safeguard_evaluation, write_csv, SAFEGUARD_TEST_QUERIES
import json

with open('/content/finance-guidance-rag/eval/test_questions.json') as f:
    test_set = json.load(f)

print("--- Retrieval evaluation ---")
accuracy, avg_latency, retrieval_rows = run_retrieval_evaluation(test_set, index, embed_model, corpus)
print(f"Retrieval accuracy: {accuracy:.1%}")
print(f"Average latency: {avg_latency*1000:.1f} ms")
write_csv(retrieval_rows, '/content/finance-guidance-rag/eval/retrieval_results.csv')

import psutil
print(f"RAM: {psutil.virtual_memory().percent}%")